In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv

# Load Credentials Securely 
# You MUST create a local .env file with your PG_... and GEMINI_API_KEY
load_dotenv() 

# Replace with environment variables
db_user = os.getenv("PG_USER", "postgres")
db_password = os.getenv("PG_PASSWORD", "admin")
db_host = os.getenv("PG_HOST", "localhost") 
db_port = os.getenv("PG_PORT", "5432")
db_name = os.getenv("PG_DB_NAME", "db_anushka")

# Create the database engine
PG_URI = f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
try:
    engine = create_engine(PG_URI)
    print("PostgreSQL connection engine created successfully.")
except Exception as e:
    print(f"Error connecting to PostgreSQL: {e}")
    engine = None

# Extract Schema for all tables
formatted_schema = ""
if engine:
    inspector = inspect(engine)
    # List the tables
    target_tables = ['departments', 'assets', 'dept_inventory', 'maintenance_log']
    
    for table_name in target_tables:
        columns = inspector.get_columns(table_name)
        
        # Format columns as: column_name (data_type)
        column_details = [f"{col['name']} ({col['type']})" for col in columns]
        
        # Add to the full schema string
        formatted_schema += f"Table '{table_name}' has columns: {', '.join(column_details)}. "
    
    print("\nSuccessfully extracted multi-table schema.")
    print("--- Example Schema Snippet ---")
    print(formatted_schema[:200] + "...") # Print a snippet for verification
else:
    formatted_schema = "Error: Database connection failed. Cannot proceed."

PostgreSQL connection engine created successfully.

Successfully extracted multi-table schema.
--- Example Schema Snippet ---
Table 'departments' has columns: dept_id (VARCHAR(10)), dept_name (VARCHAR(100)), annual_budget_usd (NUMERIC(15, 2)), building (VARCHAR(100)). Table 'assets' has columns: asset_id (VARCHAR(10)), asset...


In [5]:
# SQL Generation Function
import google.generativeai as genai

def generate_sql(user_input: str):
    # System Instruction (ensures consistent quality)
    SYSTEM_INSTRUCTION = (
        "You are an expert PostgreSQL SQL query generator. "
        "Your output MUST ONLY be the complete SQL query (no backticks, no markdown, no explanation, and no ending semicolon). "
        "Use standard JOINs across the multiple tables provided in the schema context when necessary to fulfill the request. "
        "Use table and column names exactly as provided in the schema."
    )
    
    prompt = f"""
    {SYSTEM_INSTRUCTION}
    
    SCHEMA INFO: {formatted_schema}
    
    USER REQUEST: {user_input}
    
    TASK: Based on the SCHEMA INFO, write the single PostgreSQL query to fulfill the USER REQUEST.
    """
    
    try:
        response = model.generate_content(prompt)
        final_query = response.text.strip().replace('`', '').replace(';', '')
        
        if final_query.lower().startswith('sql'):
            final_query = final_query[3:].strip()
            
        print("✅ SQL Query Generated.")
        return final_query
    except Exception as e:
        print(f"❌ Error during SQL Generation: {e}")
        return None

In [7]:
# SQL Execution Function
import pandas as pd
from IPython.display import display, Markdown

def execute_sql(final_query: str):
    df = None
    
    print(f"\n--- Executing Query ---")
    display(Markdown(f"```sql\n{final_query}\n```"))

    if engine:
        try:
            if final_query.upper().startswith(('SELECT', 'WITH')):
                df = pd.read_sql(final_query, engine)
                print(f"✅ SQL Executed. Data retrieved ({len(df) if df is not None else 0} rows).")
            else:
                print("❌ Error: Generated query is not a valid SELECT statement.")
                
        except Exception as e:
            print(f"❌ --- EXECUTION ERROR ---")
            print(f"Error Details: {e}")
            df = None
            
    return df

In [9]:
# Analytical Reporting and Display Function
from IPython.display import display, Markdown

def generate_report_and_display(df, user_input: str):
    analytical_report = "No data was returned for analysis or a critical error occurred."
    
    if df is not None and not df.empty:
        results_str = df.head(20).to_markdown(index=False)
        report_prompt = f"""
        You are an Inventory Analyst. Analyze the following data generated from the user's request: '{user_input}'.
        
        Raw Data Sample (Top {len(df.head(20))} rows):
        {results_str}
        
        Provide a concise, one-paragraph analytical summary and conclusion based on this data. 
        """
        try:
            report_response = model.generate_content(report_prompt)
            analytical_report = report_response.text
            print("✅ Analytical Report Generated.")
        except Exception as e:
            analytical_report = f"Error during report generation: {e}"
            print(f"❌ Error during Report Generation: {e}")
    else:
        print("--- Skipping Report Generation (No Data) ---")

    # Final Output Display
    print("\n" + "="*70 + "\n")
    display(Markdown(f"## 🤖 Gemini Analytical Report for: *{user_input}*"))
    display(Markdown(analytical_report))
    print("\n" + "="*70 + "\n")
    print(f"Raw Data Output (Total {len(df) if df is not None else 0} Rows)")
    if df is not None:
        display(df)

In [17]:
# FINAL EXECUTION CELL

#print("GenAI Analytical Report Engine Activated")

# 1. GET INPUT HERE (The single interactive step)
user_prompt = input("What analytical report would you like to run? ")

# 2. Chain the functions together in sequence
generated_sql = generate_sql(user_prompt)

if generated_sql:
    result_df = execute_sql(generated_sql)
    generate_report_and_display(result_df, user_prompt)

print("Engine process complete. Run this cell again for a new query.")

What analytical report would you like to run?  Show all assets in the 'Architecture' department that are currently marked 'Maintenance'. Include the asset name and the room location.


✅ SQL Query Generated.

--- Executing Query ---


```sql
SELECT
  a.asset_name,
  di.room_location
FROM assets AS a
JOIN dept_inventory AS di
  ON a.asset_id = di.asset_id
JOIN departments AS d
  ON di.dept_id = d.dept_id
WHERE
  d.dept_name = 'Architecture' AND di.status = 'Maintenance'
```

✅ SQL Executed. Data retrieved (2 rows).
✅ Analytical Report Generated.




## 🤖 Gemini Analytical Report for: *Show all assets in the 'Architecture' department that are currently marked 'Maintenance'. Include the asset name and the room location.*

This data represents a comprehensive list of all assets within the 'Architecture' department that are currently designated for 'Maintenance'. The provided information specifically identifies a "3D Printer (Resin)" located in the "Fab Workshop" and a "Plotter (A0 Inkjet)" in the "Plotting Room" as the primary assets requiring immediate attention. This indicates that key fabrication and printing equipment, essential for architectural design and production, are currently out of service or undergoing scheduled maintenance. The concise nature of the list suggests that the maintenance effort for the Architecture department is focused on these two critical pieces of hardware.



Raw Data Output (Total 2 Rows)


,asset_name,room_location
0,3D Printer (Resin),Fab Workshop
1,Plotter (A0 Inkjet),Plotting Room


Engine process complete. Run this cell again for a new query.
